# 04 — Pixel explorer

Shows the time series (VV, VH, VH − VV, rain, platforms) of a single 10 m pixel, reading only that pixel from the stack. How to read the curves: `docs/01_sar_basics.md` §5.

> **Safe to re-run:** finished work is skipped. If the kernel dies or a cell crashes, just run the same cells again.

## Setup

**Which config is used?** After a run is chosen, everything uses the run's **frozen** config (`runs/<run_id>/run_config.yaml`), so this run's files are always read with the settings that produced them. If you edited your own config since `new-run`, a warning lists the differing keys; such changes need a **new run**. `auth.project` also stays the run's project. Taken from your config instead: `qa.acknowledged_issues` (accepting QA issues is a decision made after the run), `auth.key_file` and `resources` (they describe the machine, so a run exported on a laptop can be continued on a cloud notebook server).

In [ ]:
from pathlib import Path
import sys

# Project root = parent of notebooks/. Adding src/ is only needed if you did not run `pip install -e .`
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# >>> Change this to YOUR config file (copied from config/pipeline.example.yaml; it must live in config/) <<<
CONFIG_PATH = "config/my_aoi_season.yaml"

from sar_pipeline import config, resources
user_cfg = config.load_config(ROOT / CONFIG_PATH)   # your editable config; the RUN config is loaded below
print("Config :", CONFIG_PATH)
print("Machine:", resources.detect_resources(user_cfg).describe())

from sar_pipeline.cli import load_run_context

RUN_ID = None   # None = newest run; or a run folder name such as "v001_20260915"
cfg, run_path = load_run_context(user_cfg, RUN_ID)   # frozen run config from here on
print("Run:", run_path)
print("Tracks:", [t["track_id"] for t in cfg["s1"]["tracks"]])

## Pick a pixel

- **`PID`**: pixel id from QGIS: open `processed/<aoi>/<season>/grid/pixel_index.tif`, click a field with the Identify tool, band 1 is the pid.
- **or `LON`/`LAT`**: used only when `PID` is `None`.
- Both `None`: a point inside the AOI is chosen automatically.

`TRACKS` = every selected track (one table and one plot per track). `PLOT_BACKEND` = `"matplotlib"` (static image) or `"plotly"` (interactive; needs `pip install plotly` in the kernel's environment). Do not commit the notebook with ids, coordinates or outputs: clear outputs first.

In [ ]:
import geopandas as gpd
import rasterio
from sar_pipeline import grid, index, pixel_query

PID = None              # <- pixel id from QGIS (pixel_index.tif, band 1), e.g. 123456
LON, LAT = None, None   # <- or a lon/lat point; only used when PID is None
TRACKS = [t["track_id"] for t in cfg["s1"]["tracks"]]   # or e.g. ["RO123_ASC"]
PLOT_BACKEND = "matplotlib"   # "matplotlib" (static image) or "plotly" (interactive: zoom + hover)

gd = grid.load_grid(cfg)
if PID is None:
    if LON is None or LAT is None:
        p = gpd.read_file(config.aoi_path(cfg)).to_crs("EPSG:4326").union_all().representative_point()
        LON, LAT = p.x, p.y
    PID = index.lonlat_to_pid(gd, LON, LAT)

row, col = index.pid_to_rowcol(gd, PID)
lon, lat = index.pid_to_lonlat(gd, PID)
with rasterio.open(config.grid_dir(cfg) / "pixel_index.tif") as ds:
    inside_aoi = ds.read(3, window=((row, row + 1), (col, col + 1)))[0, 0] == 1
print(f"pid {PID} | row {row} col {col} | centre lon={lon:.6f} lat={lat:.6f} | inside AOI: {inside_aoi}")
if not inside_aoi:
    print("WARNING: this pixel is outside the AOI polygon (grid margin); pick a pixel with aoi_mask = 1.")

## Time series and plot

Columns: `<POL>_db` per polarisation, `VH_minus_VV_db`, rain, platforms and `valid`. Only this one pixel is read from the stack.

In [ ]:
import pandas as pd

def plot_plotly(df, title):
    """Interactive version of pixel_query.plot_timeseries: same lines, rain marks, events and gap shading."""
    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
    except ImportError as exc:
        raise ImportError("Plotly is not installed in this kernel. Run `pip install plotly`, "
                          "or set PLOT_BACKEND = \"matplotlib\".") from exc

    audit = cfg.get("audit", {}) or {}
    t = pd.to_datetime(df["datetime_utc"]).dt.tz_localize(None)
    rain = pd.to_numeric(df["rain_24h_mm"], errors="coerce")
    info = df["acquisition_id"] + " | platform " + df["platforms"].astype(str) + " | rain 24 h " + rain.round(1).astype(str) + " mm"

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for col, color in [("VV_db", "#1f77b4"), ("VH_db", "#d62728")]:
        if col in df:
            fig.add_trace(go.Scatter(x=t, y=df[col], name=col, mode="lines+markers", line=dict(color=color),
                                     customdata=info, hovertemplate="%{y:.2f} dB<br>%{customdata}"), secondary_y=False)
    if "VH_minus_VV_db" in df:
        fig.add_trace(go.Scatter(x=t, y=df["VH_minus_VV_db"], name="VH - VV", mode="lines+markers",
                                 line=dict(color="#2ca02c", dash="dash"), hovertemplate="%{y:.2f} dB"), secondary_y=True)

    wet = rain >= 5
    if wet.any():
        y_low = min(df[c].min() for c in ("VV_db", "VH_db") if c in df) - 1
        fig.add_trace(go.Scatter(x=t[wet], y=[y_low] * int(wet.sum()), name="rain >= 5 mm / 24 h", mode="markers",
                                 marker=dict(symbol="triangle-down", size=11, color="#17becf"),
                                 customdata=rain[wet].round(1), hovertemplate="rain %{customdata} mm"), secondary_y=False)

    max_gap = float(audit.get("max_gap_days", 12))   # grey = gap longer than this: crop changes may be missed
    for a, b in zip(t.iloc[:-1], t.iloc[1:]):
        if (b - a).days > max_gap:
            fig.add_vrect(x0=a, x1=b, fillcolor="grey", opacity=0.15, line_width=0)
    for ev in audit.get("constellation_events", []) or []:   # possible sensor step
        x = pd.Timestamp(ev["date"])
        fig.add_shape(type="line", x0=x, x1=x, y0=0, y1=1, yref="paper", line=dict(color="black", dash="dot", width=1))
        fig.add_annotation(x=x, y=1, yref="paper", text=ev.get("note", ""), textangle=-90, showarrow=False,
                           xanchor="right", yanchor="top", font=dict(size=9))

    fig.update_layout(title=title, hovermode="x unified", height=480, legend=dict(orientation="h", y=-0.15))
    fig.update_yaxes(title_text="Backscatter (dB)", secondary_y=False)
    fig.update_yaxes(title_text="VH - VV (dB)", secondary_y=True)
    fig.show()

In [ ]:
import matplotlib.pyplot as plt

for track in TRACKS:
    df = pixel_query.pixel_timeseries(cfg, run_path, track, PID)
    display(df.round(2))
    title = f"{track}  pid {PID}"
    if PLOT_BACKEND == "plotly":
        plot_plotly(df, title)
    elif PLOT_BACKEND == "matplotlib":
        pixel_query.plot_timeseries(df, cfg=cfg, title=title)
        plt.show()
    else:
        raise ValueError('PLOT_BACKEND must be "matplotlib" or "plotly"')

## Reading the plot

- **VH** rising over weeks then falling → a crop cycle. For rice look for a deep **dip first** (the flooded field) and then the rise; the length separates rice/maize (~4 months) from sugarcane (~10–12 months).
- **Very low VV and VH** early in the season → flooded field (rice).
- A **single-date spike** with rain ≥ 5 mm in 24 h → weather, not growth.
- The first and last dates are noisier (fewer neighbours for the speckle filter; see `n_temporal_neighbors` in `dates.csv`).
- Vertical markers = Sentinel-1 constellation events (possible sensor step); shaded = long data gaps.